<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/ct_concept_steering_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Concept-Direction Steering Demo

Demonstrates interpretune's **concept-direction-mediated, sign-aware, multi-feature steering** on a
trivial example, `orange` color-vs-fruit sense disambiguation to familiarize the user with some of interpretune's
intervention mechanisms:

1. **Feature-mediated path**: concept direction -> attribution graph -> sign-aware
   `FeatureSelectionSpec` top-feature selection -> `feature_intervention_forward` (circuit-tracer
   feature interventions with sign-aware, influence-normalized scaling).
2. **Direct-hook path**: the same concept direction applied via `model_fwd_intervention`
   (hook-tensor add/project interventions at canonical hook points).

Both paths derive the concept direction from the token-embedding basis (`paired_rejection`
over the concept groups).

The `BACKEND` parameter selects the circuit-tracer backend for all steps, and NNsight is the one the
notebook tests validate. circuit-tracer's TransformerLens backend is unavailable for now: it builds its replacement model on
`HookedTransformer`, which TransformerLens 4.0 removes, so it returns once circuit-tracer ports that model
onto `TransformerBridge` (see `docs/circuit_tracer_backend_support.md`).

This notebook runs `gemma-2-2b` + the Gemma Scope `gemmascope-transcoder-16k` set against the
public [neuronpedia.org](https://www.neuronpedia.org) dashboards, so every selected feature's
semantics can be inspected by clicking through — no local services required. The substrate is a
base model with transcoders trained on that base model.

Running against a local Neuronpedia dev webapp instead — with locally generated feature
explanations and the instruction-tuned `gemma-3-1b-it` substrate — is a separate notebook:
[`ct_concept_steering_demo_local_np.ipynb`](ct_concept_steering_demo_local_np.ipynb).

> **Prerequisites**: a GPU with bf16 support, plus access to the model and transcoder weights.
> Nothing else — dashboard links and explanations come from the public Neuronpedia API.

> **Expected result**: the two steering paths are not equally strong. The attribution-graph
> feature-mediated path (step 2) is the one selecting features *for their causal effect on the target
> logit difference*, and it should produce the largest target-gap shift. Direct-hook steering (step 4)
> applies a concept direction at a hook point without that per-feature attribution, so it is expected
> to be weaker. That gap is a finding, not a defect: step 5's decoupling analysis exists to explain it.


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
BACKEND = "nnsight"  # circuit-tracer backend for all steps: "nnsight" or "transformerlens"
CONCEPT_PROMPT = "Is orange a color or a fruit? Answer with one word: Color or Fruit. orange ->"
CONCEPT_TARGET_TOKENS = ["Fruit", "Color"]
FEATURE_SELECTION_TOP_N = 5
FEATURE_SELECTION_MIN_LAYER = 10  # fs_l10_n5 lineage: layers >= 10
FEATURE_SELECTION_SCORE_SIGN = "any"  # any | positive | negative
INTERVENTION_SCALE_FACTOR = 20.0  # validated s5_any demo scale
EMBED_INTERVENTION_OP = "add"  # model_fwd_intervention op for the embed step
EMBED_INTERVENTION_HOOK = "unembed.hook_in"

# -- Model / dashboard substrate: public gemma-2-2b + public neuronpedia.org dashboards ---------
# Base model + base-trained transcoders, so every selected feature can be inspected directly on
# neuronpedia.org. The local-Neuronpedia substrate lives in ct_concept_steering_demo_local_np.ipynb.
REGISTRY_KEY = "prompt_analysis.gemma2.circuit_tracer"  # hub component configuration key (model + backend)
MODEL_NAME = "gemma-2-2b"
TRANSCODER_SET = "gemma"  # circuit-tracer transcoder set override (None keeps the registry default)
NEURONPEDIA_MODEL_ID = "gemma-2-2b"
NEURONPEDIA_SOURCE_SET = "gemmascope-transcoder-16k"
CHAT_FORMAT_PROMPT = False  # True for instruction-tuned models (render CONCEPT_PROMPT via chat template)

# -- Section 4b (J-space patch steering) --------------------------------------------------------
JLENS_OPS_REPO = "speediedan/jlens_steering_ops"  # private until #261 flips it public; needs HF auth with access
JLENS_OPS_REVISION = "75d40196479048a94a80699a4da7abc218e0663d"  # pinned: trusted hub code must not change under us
JLENS_LAYER = 24  # fitted lens layer; later layers suit naming (gemma-2-2b flips at 24, the last fitted)
JLENS_PATCH_SCALE = 1.0  # a pure coordinate swap -- the J-space path needs no oversteering
SCALE_SWEEP = (1.0, 5.0, 20.0)  # shared multipliers for the 4b scaling curve
RUN_JSPACE_SECTION = True  # skips cleanly (with the reason) when the collection is unreachable

In [3]:
# @title Imports { display-mode: "form" }
import torch  # noqa: F401

import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis.backends import FeatureSelectionSpec  # noqa: F401
from it_examples.utils.nb_ui_utils import (  # noqa: F401
    best_variant_token_ids,
    display_jlens_readout_comparison,
    display_steering_results,
    display_target_gap,
    display_top_features_comparison,
    resolve_feature_explanations,
)

## 1. Session setup

Single-backend circuit-tracer session built from the `REGISTRY_KEY` hub component configuration
(default: `prompt_analysis.gemma2.circuit_tracer` with the Gemma Scope transcoder set that matches the
public `gemma-2-2b` dashboards).

> **Note:** resolution is cache-backed and per-key: `it.hub.load` hydrates only the requested configuration.


In [4]:
# @title 1: Session construction { display-mode: "form" }
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it
from interpretune import ITSession, ITSessionConfig

# load HF credentials before session init (model + transcoder downloads)
for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/prompt-analysis", REGISTRY_KEY)
# single circuit-tracer backend for all phases (BACKEND selects the replacement-model implementation)
base_it_cfg.circuit_tracer_cfg.backend = BACKEND
if TRANSCODER_SET:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = TRANSCODER_SET
if BACKEND == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    # the TL circuit-tracer backend needs the transformer_lens adapter in the composition
    # (it provides the replacement-model init path; see docs/circuit_tracer_backend_support.md)
    adapter_ctx = (it.Adapter.core, it.Adapter.transformer_lens, it.Adapter.circuit_tracer)
session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
tokenizer = module.replacement_model.tokenizer
print(f"session ready: {type(module).__name__} ({MODEL_NAME} + circuit-tracer {BACKEND} backend)")

src/interpretune/config/shared.py:281: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'tl_cfg': ITLensFromPretrainedNoProcessingConfig(move_to_device=True, default_padding_side='left', model_name='gemma-2-2b', fold_ln=False, center_writing_weights=False, center_unembed=False, refactor_factored_attn_matrices=False, checkpoint_index=None, checkpoint_value=None, hf_model=None, device='cuda', n_devices=1, tokenizer=None, fold_value_biases=False, default_prepend_bos=True, dtype='float32'), 'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name=None, transcoder_set='gemma', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=8192, offload='cpu', freeze_attention=True, freeze_norms=True, lazy_encoder=None, lazy_decoder=True, lazy_delta_rank=None, tune_base=False, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True,

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer


... (2 outputs left out)

src/interpretune/session.py:322: CircuitTracerModule is not interpretune.protocol.BuiltTrainSteppable | interpretune.protocol.BuiltValidationSteppable | interpretune.protocol.BuiltTestSteppable | interpretune.protocol.BuiltPredictSteppable even after composing <class 'interpretune.adapters.circuit_tracer.adapter.CircuitTracerModule'> with <class 'interpretune.session.InterpretunableModule'>. This Interpretune session may not work as intended.
[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


session ready: InterpretunableModule (gemma-2-2b + circuit-tracer nnsight backend)


## 2. Feature-mediated path: concept direction -> attribution -> sign-aware selection -> feature steering

Runs the registered composite `it.intervention_from_concept(...)` (concept_direction ->
compute_attribution_graph -> graph_node_influence -> extract_top_features ->
feature_intervention_forward) with:

- `FeatureSelectionSpec(layer_slice=(FEATURE_SELECTION_MIN_LAYER, None), score_sign=FEATURE_SELECTION_SCORE_SIGN,
  score_source="signed_influence")`
- sign-aware, influence-normalized scaling
  (`intervention_sign_aware_scale=True`, `intervention_max_influence_norm_scale=True`,
  `intervention_value_source="top_feature_activation_values"`, `intervention_scale_factor=INTERVENTION_SCALE_FACTOR`)

Expected outcome: post-intervention target gap exceeds the pre-intervention gap and the
post-intervention argmax lands in the target-token variant set.


In [5]:
# @title 2: Feature-mediated steering { display-mode: "form" }
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.config import AnalysisCfg, init_analysis_cfgs

module.analysis_cfg = AnalysisCfg(target_op=it.compute_attribution_graph, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

fruits = ["apple", "banana", "grape", "peach"]
colors = ["red", "blue", "green", "yellow"]
if CHAT_FORMAT_PROMPT:
    # instruction-tuned replacement models assert chat-formatted inputs
    from it_examples.examples.prompt_configs.prompt_configs import GemmaPromptConfig

    prompt = GemmaPromptConfig().apply_chat_template_fn(
        tokenizer, CONCEPT_PROMPT, tokenize=False, add_generation_prompt=True
    )
else:
    prompt = CONCEPT_PROMPT

# sign-aware, influence-normalized scaling (the validated s5_any lineage)
ct_cfg = module.it_cfg.circuit_tracer_cfg
ct_cfg.intervention_sign_aware_scale = True
ct_cfg.intervention_max_influence_norm_scale = True
ct_cfg.intervention_value_source = "top_feature_activation_values"

selection_spec = FeatureSelectionSpec(
    layer_slice=slice(FEATURE_SELECTION_MIN_LAYER, None),
    score_source="signed_influence",
    score_sign=FEATURE_SELECTION_SCORE_SIGN,
    rank_by_abs=True,
)
pipeline_results = it.intervention_from_concept(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color",
        concept_direction_mode="paired_rejection",
        concept_basis="embed",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=FEATURE_SELECTION_TOP_N,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
    feature_selection=selection_spec,
)
# one call renders the linked/signed features table + the consolidated target-gap table and
# returns everything later phases need (features, direction, target ids, gaps)
DASHBOARD_BASE_URL = "https://www.neuronpedia.org"
steering_base_url = DASHBOARD_BASE_URL
steering = display_steering_results(
    pipeline_results,
    tokenizer,
    CONCEPT_TARGET_TOKENS,
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
    neuronpedia_base_url=steering_base_url,
    min_layer=FEATURE_SELECTION_MIN_LAYER,
)
steered_features = steering.steered_features
pipeline_direction = steering.direction
target_a_id, target_b_id = steering.target_ids
fm_pre_gap, fm_post_gap = steering.pre_gap, steering.post_gap
assert fm_post_gap > fm_pre_gap, "feature-mediated steering should push the gap toward the target concept"

Phase 0: Precomputing activations and vectors


Precomputation completed in 1.63s


Found 18348 active features


Phase 1: Running forward pass


Forward pass completed in 0.21s


Phase 2: Building input vectors


... (40 outputs left out)


Feature attributions completed in 3.85s


Attribution completed in 8.87s


#,Node,Sign,|Score|
1,"(25, 19, 16131)",−,1.94e-08
2,"(24, 19, 13277)",−,1.81e-08
3,"(24, 19, 5999)",+,9.92e-09
4,"(24, 19, 3865)",+,4.87e-09
5,"(25, 19, 13210)",+,4.74e-09


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,1.879%,25.1250,29.3750,+4.2500
Color,15.106%,1.04e-05,27.0000,21.8750,-5.1250
Gap (Fruit − Color),,,-1.8750,+7.5000,+9.3750


## 3. Feature semantics: the top-features table

The steered features render as a table with the `(layer, pos, feature)` node tuple linked to its
dashboard, the **signed** influence score (Sign / |Score| columns — the `signed_influence`
selection can steer with negative-signed features, so the sign is colour-coded), and a best-effort
**Explanation** column resolved from the public Neuronpedia feature API.


In [6]:
# @title 3: Top-features table { display-mode: "form" }
# top_feature_ids are (layer, position, feature) tuples; dashboards/explanations are per
# (layer, feature), so collapse positions while preserving selection order
steered_layer_feature_pairs = list(dict.fromkeys((f[0], f[-1]) for f in steered_features))

# Best-effort explanation text from the public neuronpedia.org feature API; unmapped features
# simply render an empty Explanation cell
feature_explanations = resolve_feature_explanations(
    model_id=NEURONPEDIA_MODEL_ID,
    source_set=NEURONPEDIA_SOURCE_SET,
    feature_tuples=steered_layer_feature_pairs,
    base_url=DASHBOARD_BASE_URL,
)

display_top_features_comparison(
    {"Steered Features (signed influence)": steered_features},
    {"Steered Features (signed influence)": pipeline_results.top_feature_scores.tolist()},
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
    neuronpedia_base_url=DASHBOARD_BASE_URL,
    show_score_sign=True,
    feature_explanations=feature_explanations,
)

#,Node,Sign,|Score|,Explanation
1,"(25, 19, 16131)",−,1.94e-08,"the comparison of two varieties of fruit, relating to size, taste, color, and genetic information"
2,"(24, 19, 13277)",−,1.81e-08,words related to questions and requests
3,"(24, 19, 5999)",+,9.92e-09,"language related to institutions, negative situations, the internet, and programming languages"
4,"(24, 19, 3865)",+,4.87e-09,"dollar signs and other currency symbols, potentially alongside numbers or related terms like ""terms"" and ""bonus""."
5,"(25, 19, 13210)",+,4.74e-09,grammatical structures and parts of speech like noun phrases and verb phrases


## 4. Direct-hook path: concept direction -> hook-tensor steering

Recomputes the embed-basis concept direction for the same concept pair (a consistency check against
the step 2 pipeline's direction — cosine should be ~1.0 since both derive from the same
embedding-basis `paired_rejection`) and applies `it.model_fwd_intervention(...)` at
`EMBED_INTERVENTION_HOOK` with the `EMBED_INTERVENTION_OP` op, comparing
`pre/post_intervention_logits` and the target-token gap against the feature-mediated result. The
same direction steered through selected transcoder features vs added directly at the hook point
produces different effect sizes — the feature-mediated path is typically stronger per unit scale.


In [7]:
# @title 4: Direct-hook steering { display-mode: "form" }
# Recompute the embed-basis concept direction for the same concept pair (no store rows -> embed basis)
direct_result = it.concept_direction(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color (direct)",
        concept_direction_mode="paired_rejection",
        concept_basis="embed",
    ),
    None,
    0,
)
direct_direction = direct_result.concept_direction.detach()
cosine = torch.nn.functional.cosine_similarity(
    pipeline_direction, direct_direction.float().cpu().reshape(-1), dim=0
).item()
print(f"pipeline-vs-direct direction cosine: {cosine:+.4f} (~1.0 expected — same embed-basis construction)")

# Direct hook-tensor intervention at the canonical hook point
module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

# chat-rendered prompts already carry their special tokens; plain completion prompts need them added
enc = tokenizer(prompt, return_tensors="pt", padding=False, add_special_tokens=not CHAT_FORMAT_PROMPT)
device = next(module.model.parameters()).device
if BACKEND == "transformerlens":
    # HookedTransformer.forward takes `input`, not the HF-style `input_ids`/`attention_mask` keys
    batch = {"input": enc["input_ids"].to(device)}
else:
    batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in dict(enc).items()}

# legacy HookedTransformer models (the CT TransformerLens backend) expose no `unembed.hook_in`;
# their pre-unembed equivalent is `ln_final.hook_normalized` (alias-map expansion tracked in
# interpretune#223)
intervention_hook = EMBED_INTERVENTION_HOOK
if BACKEND == "transformerlens" and EMBED_INTERVENTION_HOOK == "unembed.hook_in":
    intervention_hook = "ln_final.hook_normalized"

direct_batch = AnalysisBatch(
    prompts=[prompt],
    concept_direction=direct_direction,
    logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
    concept_group_a_token_ids=[target_a_id],
    concept_group_b_token_ids=[target_b_id],
    concept_cache_key=intervention_hook,
    intervention_hook_pattern=intervention_hook,
    intervention_op=EMBED_INTERVENTION_OP,
    direction_scale_factor=INTERVENTION_SCALE_FACTOR,
)
direct_out = it.model_fwd_intervention(module, direct_batch, batch, 0)

direct_pre_gap, direct_post_gap = display_target_gap(
    direct_out.pre_intervention_logits.float().cpu().reshape(-1),
    direct_out.post_intervention_logits.float().cpu().reshape(-1),
    (CONCEPT_TARGET_TOKENS[0], target_a_id),
    (CONCEPT_TARGET_TOKENS[1], target_b_id),
    title="Direct-hook steering — target gap",
)
print(
    f"feature-mediated delta {fm_post_gap - fm_pre_gap:+.3f} vs direct-hook delta "
    f"{direct_post_gap - direct_pre_gap:+.3f}"
)
assert direct_post_gap > direct_pre_gap, "direct-hook steering should push the gap toward the target concept"

pipeline-vs-direct direction cosine: +1.0000 (~1.0 expected — same embed-basis construction)


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,2.036%,25.1250,27.8750,+2.7500
Color,15.106%,1.797%,27.0000,27.7500,+0.7500
Gap (Fruit − Color),,,-1.8750,+0.1250,+2.0000


feature-mediated delta +9.375 vs direct-hook delta +2.000


In [8]:
# @title 4a: Embed-basis patch steering { display-mode: "form" }
# The patch primitive is basis-agnostic: the same coordinate swap run along an
# embed-basis pole pair, beside section 4's naive add. Poles are per-group mean
# token embeddings; with the pair spanning the concept axis, the swap preserves
# everything orthogonal to it -- and the share below quantifies what the naive
# add moves outside that axis.
from interpretune.analysis.optools import resolve_unembed_and_norm_scale
from it_examples.utils.steering_demo_helpers import (
    embed_pole_pair,
    final_norm_output,
    outside_span_share,
    pole_swap_prediction,
)

_ep_V = embed_pole_pair(module, tokenizer, fruits, colors)
_ep_orth_share = outside_span_share(direct_direction, _ep_V)

module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])
_ep_batch = AnalysisBatch(
    prompts=[prompt],
    concept_direction=direct_direction,
    logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
    concept_group_a_token_ids=[target_a_id],
    concept_group_b_token_ids=[target_b_id],
    concept_cache_key=intervention_hook,
    intervention_hook_pattern=intervention_hook,
    intervention_op="patch",
    intervention_tensor=_ep_V,
    direction_scale_factor=1.0,
)
_ep_out = it.model_fwd_intervention(module, _ep_batch, batch, 0)
_ep_pre_gap, _ep_post_gap = display_target_gap(
    _ep_out.pre_intervention_logits.float().cpu().reshape(-1),
    _ep_out.post_intervention_logits.float().cpu().reshape(-1),
    (CONCEPT_TARGET_TOKENS[0], target_a_id),
    (CONCEPT_TARGET_TOKENS[1], target_b_id),
    title="Embed-basis patch steering — target gap",
)
print(
    f"embed-basis patch delta {_ep_post_gap - _ep_pre_gap:+.3f} (scale 1.0) vs "
    f"naive add delta {direct_post_gap - direct_pre_gap:+.3f} (scale {INTERVENTION_SCALE_FACTOR}); "
    f"add moves {_ep_orth_share:.1%} of its norm outside the pole-pair span"
)
# A swap exchanges the clean activation's two pole coordinates, so its DIRECTION is set by which pole the clean
# state already leans toward, not by whether the basis is right: it pushes toward Fruit only when the clean state
# sits nearer the color pole. This site is the unembed's input, after the final norm, so the folded/unfolded choice
# does not arise here; see "Sites after the final norm" in
# https://interpretune.org/en/latest/usage/jlens_norm_folding.html. So the check is the swap's first-order prediction
# from the clean coordinates (linear in the logits), not an expected direction per model.
_ep_info = resolve_unembed_and_norm_scale(module)
_ep_pred = pole_swap_prediction(
    final_norm_output(module, batch), _ep_V, _ep_info.w_u[target_a_id], _ep_info.w_u[target_b_id]
)
print(
    f"clean pole coordinates: fruit {_ep_pred.coordinates[0]:+.3f}, color {_ep_pred.coordinates[1]:+.3f}; "
    f"predicted swap delta {_ep_pred.predicted_gap_delta:+.3f}, measured {_ep_post_gap - _ep_pre_gap:+.3f}"
)
assert (_ep_post_gap > _ep_pre_gap) == (_ep_pred.predicted_gap_delta > 0), "swap moved against its prediction"

Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,5.535%,25.1250,26.1250,+1.0000
Color,15.106%,15.046%,27.0000,27.1250,+0.1250
Gap (Fruit − Color),,,-1.8750,-1.0000,+0.8750


embed-basis patch delta +0.875 (scale 1.0) vs naive add delta +2.000 (scale 20.0); add moves 4.8% of its norm outside the pole-pair span
clean pole coordinates: fruit +7.422, color +12.992; predicted swap delta +3.673, measured +0.875


## 4b. J-space path: workspace patch steering (hub op collection)

Steers the same concept pair a third way, in **J-space** (Gurnee et al. 2026, the [workspace
paper](https://transformer-circuits.pub/2026/workspace/)), using a **genuinely non-bundled op
collection** pulled from the Hub at run time: `JLENS_OPS_REPO` builds the `(2, d_model)` J-lens pole
pair (`v_c = (W_U[c] * final_norm_scale) @ J_l`, lenses pre-fitted in
[`neuronpedia/jacobian-lens`](https://huggingface.co/neuronpedia/jacobian-lens)) and stages a
lens-coordinate `patch` intervention, `h <- h + V(sigma(c) - c)` -- a coordinate **swap** that leaves
everything orthogonal to the pair untouched.

Contrast with the two paths above at each path's own scale: scale
conventions are not comparable across modes, so the page does not
attempt a matched-displacement comparison. On gemma-2-2b, the artifact shows
feature-mediated delta +9.375 and direct-hook add delta +2.000 (both scale
20.0) against the J-space patch delta +4.500 (scale 1.0, basis
jlens_folded). The swap's virtue is surgicality -- a coordinate exchange
that leaves everything orthogonal to the pair untouched -- a structural claim
about the construction, not an efficiency claim: the `add` at the swap's own
displacement, which would separate the two, is not computed on this page.

The collection executes code from the Hub, so the cell opts into `IT_TRUST_REMOTE_CODE` and pins the
collection revision (see the [hub trust posture](https://interpretune.org/en/latest/usage/hub_trust_posture.html)).
It skips cleanly, with the reason, when the repo is unreachable (it is private until interpretune#261).
These steering directions come from a **J-lens**. On the two models validated so far, the J-lens swaps better
than J++; see [J-Space Usage](https://interpretune.org/en/latest/usage/jlens_usage.html). J++ is a readout
option, demonstrated beside a J-lens in `jspace_workspace_demo`.


In [9]:
# @title 4b: J-space patch steering { display-mode: "form" }
import os

jspace_ran, jspace_skip_reason = False, "RUN_JSPACE_SECTION=False"
if RUN_JSPACE_SECTION:
    # Hub-resident op code: opt in explicitly and pin the revision so trusted code cannot change under us.
    os.environ.setdefault("IT_TRUST_REMOTE_CODE", "1")
    try:
        _, jlens_ops_commit = it.hub.pull_ops(JLENS_OPS_REPO, revision=JLENS_OPS_REVISION)
        jspace_ran = True
    except Exception as pull_err:  # private until #261 flips it public: no access/offline -> skip with the reason
        jspace_skip_reason = f"could not pull {JLENS_OPS_REPO}: {pull_err}"

if jspace_ran:
    print(f"pulled {JLENS_OPS_REPO} @ {jlens_ops_commit[:10]}")
    module.analysis_cfg = AnalysisCfg(target_op=it.jlens_patch_intervention, ignore_manual=True, save_tokens=False)
    init_analysis_cfgs(module, [module.analysis_cfg])
    jlens_batch = AnalysisBatch(
        prompts=[prompt],
        concept_group_a_token_ids=[target_a_id],
        concept_group_b_token_ids=[target_b_id],
        logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
        jlens_layer=JLENS_LAYER,
        direction_scale_factor=JLENS_PATCH_SCALE,
    )
    jlens_out = it.jlens_patch_intervention(module, jlens_batch, batch, 0)
    jl_pre_gap, jl_post_gap = display_target_gap(
        jlens_out.pre_intervention_logits.float().cpu().reshape(-1),
        jlens_out.post_intervention_logits.float().cpu().reshape(-1),
        (CONCEPT_TARGET_TOKENS[0], target_a_id),
        (CONCEPT_TARGET_TOKENS[1], target_b_id),
        title=f"J-space patch steering — target gap "
        f"(layer {int(jlens_out.jlens_source_layer)}, basis {jlens_out.jlens_basis}, scale {JLENS_PATCH_SCALE})",
    )
    print(
        f"feature-mediated delta {fm_post_gap - fm_pre_gap:+.3f} (scale {INTERVENTION_SCALE_FACTOR}) | "
        f"direct-hook add delta {direct_post_gap - direct_pre_gap:+.3f} (scale {INTERVENTION_SCALE_FACTOR}) | "
        f"J-space patch delta {jl_post_gap - jl_pre_gap:+.3f} "
        f"(basis {jlens_out.jlens_basis}, scale {JLENS_PATCH_SCALE})"
    )
    assert jl_post_gap > jl_pre_gap, "J-space patch steering should push the gap toward the target concept"
    assert jl_post_gap > 0, "a scale-1.0 workspace swap at this layer should flip the answer outright"
else:
    print(f"[SKIPPED] J-space section: {jspace_skip_reason}")

pulled speediedan/jlens_steering_ops @ 75d4019647


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,16.075%,25.1250,26.8750,+1.7500
Color,15.106%,1.164%,27.0000,24.2500,-2.7500
Gap (Fruit − Color),,,-1.8750,+2.6250,+4.5000


feature-mediated delta +9.375 (scale 20.0) | direct-hook add delta +2.000 (scale 20.0) | J-space patch delta +4.500 (basis jlens_folded, scale 1.0)


### Scaling curve across the three arms

Each arm is re-run at the shared multipliers in `SCALE_SWEEP`. The chart plots the target gap *after* the
intervention against scale, so the dashed line at zero is where the answer flips, the same threshold for every arm
because all three start from the same clean gap. The table beneath it gives each run's shift in logits and in
probability.

The multipliers are each arm's own, not matched displacements: an `add` multiplier scales a unit direction while a
`patch` multiplier scales a coordinate swap, so equal numbers are not equal edits. The chart shows where each arm's
own knob starts to change the answer, not which edit is larger. The feature-mediated arm re-applies the
features step 2 already selected, so the sweep adds a handful of forward passes and no new attribution graph.

In [10]:
# @title 4b: scaling curve across the three arms { display-mode: "form" }
from it_examples.utils.nb_ui_utils import display_steering_scale_sweep
from it_examples.utils.steering_demo_helpers import steering_scale_point


def _sweep_feature(scale):
    module.analysis_cfg = AnalysisCfg(target_op=it.feature_intervention_forward, ignore_manual=True, save_tokens=False)
    init_analysis_cfgs(module, [module.analysis_cfg])
    out = it.feature_intervention_forward(
        module,
        AnalysisBatch(
            prompts=[prompt],
            top_feature_ids=pipeline_results.top_feature_ids,
            top_feature_scores=pipeline_results.top_feature_scores,
            top_feature_activation_values=pipeline_results.top_feature_activation_values,
            logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
        ),
        None,
        0,
        intervention_scale_factor=scale,
    )
    return out.pre_intervention_logits, out.post_intervention_logits


def _sweep_direct(scale):
    module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
    init_analysis_cfgs(module, [module.analysis_cfg])
    out = it.model_fwd_intervention(
        module,
        AnalysisBatch(
            prompts=[prompt],
            concept_direction=direct_direction,
            logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
            concept_group_a_token_ids=[target_a_id],
            concept_group_b_token_ids=[target_b_id],
            concept_cache_key=intervention_hook,
            intervention_hook_pattern=intervention_hook,
            intervention_op=EMBED_INTERVENTION_OP,
            direction_scale_factor=scale,
        ),
        batch,
        0,
    )
    return out.pre_intervention_logits, out.post_intervention_logits


def _sweep_jspace(scale):
    module.analysis_cfg = AnalysisCfg(target_op=it.jlens_patch_intervention, ignore_manual=True, save_tokens=False)
    init_analysis_cfgs(module, [module.analysis_cfg])
    out = it.jlens_patch_intervention(
        module,
        AnalysisBatch(
            prompts=[prompt],
            concept_group_a_token_ids=[target_a_id],
            concept_group_b_token_ids=[target_b_id],
            logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
            jlens_layer=JLENS_LAYER,
            direction_scale_factor=scale,
        ),
        batch,
        0,
    )
    return out.pre_intervention_logits, out.post_intervention_logits


# Each arm's own section: the scale it ran at and the gap change it measured.
_sweep_arms = {
    "feature-mediated": (_sweep_feature, INTERVENTION_SCALE_FACTOR, fm_post_gap - fm_pre_gap),
    "direct-hook add": (_sweep_direct, INTERVENTION_SCALE_FACTOR, direct_post_gap - direct_pre_gap),
}
if jspace_ran:
    _sweep_arms["J-space patch"] = (_sweep_jspace, JLENS_PATCH_SCALE, jl_post_gap - jl_pre_gap)
_sweep = [
    steering_scale_point(name, scale, *run(scale), target_a_id, target_b_id)
    for name, (run, _, _) in _sweep_arms.items()
    for scale in SCALE_SWEEP
]
display_steering_scale_sweep(
    _sweep, CONCEPT_TARGET_TOKENS, title=f"Target gap after intervention vs scale ({MODEL_NAME})"
)
# Positive control: at its own section's scale, each arm must reproduce the change that section measured, so the
# curve is the same machinery the sections ran rather than a reconstruction of it.
for _p in _sweep:
    _, _own_scale, _own_delta = _sweep_arms[_p.arm]
    if _p.scale == _own_scale:
        assert abs((_p.post_gap - _p.pre_gap) - _own_delta) < 0.05, f"{_p.arm} sweep does not reproduce its section"
if not jspace_ran:
    print(f"[SKIPPED] J-space arm of the sweep: {jspace_skip_reason}")

Arm,Scale,Δ logit Fruit,Δ logit Color,Δ gap (Fruit − Color),Δ prob Fruit (pp),Δ prob Color (pp),Answer flipped
feature-mediated,1,+2.250,+0.000,+2.250,+7.90,-8.09,yes
feature-mediated,5,+3.875,-0.875,+4.750,+9.70,-14.43,yes
feature-mediated,20,+4.250,-5.125,+9.375,-0.45,-15.11,yes
direct-hook add,1,+0.125,+0.000,+0.125,+0.25,-0.35,no
direct-hook add,5,+0.875,+0.250,+0.625,+2.00,-0.04,no
direct-hook add,20,+2.750,+0.750,+2.000,-0.28,-13.31,yes
J-space patch,1,+1.750,-2.750,+4.500,+13.76,-13.94,yes
J-space patch,5,+4.750,+2.625,+2.125,+9.98,-5.53,yes
J-space patch,20,+4.875,+3.000,+1.875,-0.89,-13.68,no


## 4c. J-lens direction validation

Checks that the J-lens pole pair accounts for the edit 4b made with it. `interpretune`'s subspace attribution
splits the first-order prediction of the patch's effect per direction: with $V = [v_s\ v_t]$, the displacement
$\Delta h$ the swap applied, the gradient $g = \partial \mathrm{gap} / \partial h$ at the lens site,
pseudoinverse coordinates $\Delta c = V^{+}\Delta h$ (the read side `patch` mode uses) and per-direction
readouts $w = V^{\top} g$, direction $i$ explains

$$
a_i = w_i \Delta c_i, \qquad \sum_i a_i + r = g^{\top}\Delta h,
$$

so the shares plus the remainder $r$ reconstruct the prediction exactly, and a dictionary that explains nothing
reports a large remainder rather than large shares. Here the dictionary is the pair that produced the
displacement, so $r$ is numerically zero and the interesting numbers are the split between the poles and the gap
between the linear prediction and the measured change (everything nonlinear downstream of the lens site).

The cell takes one forward and backward pass on the underlying model for $h$ and $g$, reuses the shared
`apply_intervention` maths for $\Delta h$, and cross-checks each readout against a central difference of the gap
through `add`-mode probes along the same direction (the two agree to the extent the gap is locally linear). The
maths lives in `it_examples.utils.steering_demo_helpers`, shared by both steering demos.

This is a validation of the J-lens directions, not a comparison with the SAE-feature path of cell 3: gap shares
per J-lens coordinate and graph influence per feature are different vocabularies, and setting them side by side
would imply a comparison this machinery cannot yet make. A like-for-like comparison waits on J-space attribution
graphs ([#683](https://github.com/speediedan/interpretune/issues/683)).

In [11]:
# @title 4c: J-lens direction validation { display-mode: "form" }
# Validates the J-lens pole pair against the edit it produced. One forward + backward gives the clean
# activation h at the lens site and the gradient g of the gap there; the shared apply_intervention maths
# gives the swap's displacement; subspace_attribution_scores splits the first-order prediction per pole.
# Central-difference slopes through add-mode probes cross-check the gradient readouts.
from interpretune.analysis.backends import InterventionSpec, apply_intervention
from interpretune.analysis.optools import jlens_direction_rows, resolve_jlens_layer, resolve_unembed_and_norm_scale
from interpretune.analysis.ops.bundled.jlens.jlens_ops import subspace_attribution_scores
from it_examples.utils.nb_ui_utils import display_jlens_direction_validation
from it_examples.utils.steering_demo_helpers import capture_site_gap_gradient, central_difference_gap_slopes

if jspace_ran:
    _info = resolve_unembed_and_norm_scale(module)
    # MODEL_NAME coincides with the lens repository's model id on this demo substrate
    _jj, _jlayer, _ = resolve_jlens_layer(module, {}, {"jlens_model_id": MODEL_NAME, "jlens_layer": JLENS_LAYER})
    _V = jlens_direction_rows(_info, [target_a_id, target_b_id], _jj, apply_norm=True).detach().float()
    _site = f"blocks.{_jlayer}.hook_resid_post"
    _pole_labels = [f"{CONCEPT_TARGET_TOKENS[0]}-pole", f"{CONCEPT_TARGET_TOKENS[1]}-pole"]

    _sg = capture_site_gap_gradient(module, batch, _jlayer, target_a_id, target_b_id)
    _h, _last = _sg.activation, _sg.last_pos
    # The swap's displacement at the last token, from the same maths 4b's patch ran.
    _spec = InterventionSpec(intervention_tensor=_V.cpu(), op="patch", scale_factor=JLENS_PATCH_SCALE)
    _delta_h = (apply_intervention(_h.clone(), _spec, last_pos=_last) - _h)[0, _last]
    _attr = subspace_attribution_scores(
        _sg.gradient[0, _last], _delta_h, _V.cpu(), [target_a_id, target_b_id], jlens_out.jlens_basis
    )
    _slopes = central_difference_gap_slopes(module, prompt, batch, _site, _V, target_a_id, target_b_id)

    _val = display_jlens_direction_validation(
        _attr,
        _pole_labels,
        finite_difference_slopes=_slopes,
        measured_delta=jl_post_gap - jl_pre_gap,
        title=f"J-lens direction validation — pole pair at layer {_jlayer}, basis {_attr['basis']}",
    )
    print(
        f"first-order prediction {_val.predicted_delta:+.3f} = explained {_val.attribution_total:+.3f} + remainder "
        f"{_val.unexplained_remainder:+.3f}; measured patch delta {jl_post_gap - jl_pre_gap:+.3f} | pole fractions "
        + ", ".join(f"{lbl} {frac:.1%}" for lbl, frac in zip(_val.direction_labels, _val.direction_fractions))
    )
    # The pair produced the displacement, so the pair explains it: a remainder that is not numerically
    # zero means the dictionary and the displacement disagree about the basis, which is a defect.
    assert abs(_val.unexplained_remainder) < 0.01 * abs(_val.predicted_delta), "pair should explain its own patch"
    # Displacement norms, per arm: concept directions are unit by construction, so a bare
    # "scale 1.0 vs scale 20" headline compares multipliers rather than edits. Verify the
    # unit norms instead of assuming them, and measure the one applied edit known exactly.
    _fm_dir_norm = float(torch.linalg.vector_norm(pipeline_direction.float()))
    _direct_dir_norm = float(torch.linalg.vector_norm(direct_direction.float()))
    assert abs(_fm_dir_norm - 1.0) < 1e-4, "fm direction left unit construction"
    assert abs(_direct_dir_norm - 1.0) < 1e-4, "direct direction left unit construction"
    _jl_applied_norm = float(torch.linalg.vector_norm(_delta_h.float()))
    print(
        f"direction norms: fm {_fm_dir_norm:.4f} (nominal scale {INTERVENTION_SCALE_FACTOR}), "
        f"direct {_direct_dir_norm:.4f} (nominal scale {INTERVENTION_SCALE_FACTOR}), "
        f"J-space applied edit {_jl_applied_norm:.4f} (scale {JLENS_PATCH_SCALE}); "
        f"fm effective scale is further modified inside the op (sign-aware + max-influence-norm)"
    )
else:
    print(f"[SKIPPED] J-lens direction validation: {jspace_skip_reason}")

first-order prediction +4.168 = explained +4.168 + remainder -0.000; measured patch delta +4.500 | pole fractions Fruit-pole 60.1%, Color-pole 39.9%
direction norms: fm 1.0000 (nominal scale 20.0), direct 1.0000 (nominal scale 20.0), J-space applied edit 41.2126 (scale 1.0); fm effective scale is further modified inside the op (sign-aware + max-influence-norm)


## 4d. J-space readout at the answer position, pre/post swap

The gap flip in 4b is measured in model logits. This cell asks whether the lens itself sees the flip: the J-space readout of the clean activation versus the swapped one at the answer position, through the same folded-norm readout the `jlens_read` op uses. No new forwards are needed — the post-swap activation is `_h` $+ \Delta h$ by construction, since the patch applies additively at the site. What is asserted is structure (finite, shaped, ranked); the movement on display IS the workspace claim, and the hard guarantees live in the unit tests, not in a demo.


In [12]:
# @title 4d: J-space readout at the answer position { display-mode: "form" }
# Pre/post J-lens readout at the answer position, through the jlens_read op's own readout (RMS divisor off:
# within one position it cannot change rankings). Post is h + Δh by construction; nothing here runs the model.
from it_examples.utils.steering_demo_helpers import jlens_readout_top_tokens

if jspace_ran:
    _pre_act = _h[0, _last]
    _post_act = _h[0, _last] + _delta_h
    assert _pre_act.shape == _post_act.shape == _jj.shape[1:], "activation/lens width mismatch"
    assert torch.isfinite(_pre_act).all() and torch.isfinite(_post_act).all(), "non-finite activation"
    _pre_top = jlens_readout_top_tokens(_pre_act, _jj, _info, tokenizer)
    _post_top = jlens_readout_top_tokens(_post_act, _jj, _info, tokenizer)
    display_jlens_readout_comparison(_pre_top, _post_top, layer=_jlayer, basis=jlens_out.jlens_basis)
else:
    print(f"[SKIPPED] J-space readout: {jspace_skip_reason}")

#,Pre token,Pre,Post token,Post,Move
1,color,+563.04,fruit,+635.48,+3
2,Color,+528.75,Fruit,+612.29,new
3,noun,+489.63,noun,+489.64,=
4,fruit,+452.37,Fruit,+486.91,new
5,orange,+447.67,FRUIT,+484.95,new


## 5. Input/output decoupling analysis (graph hydration + UMAP)

Attribution-selected steering features are chosen for their *output* effect (decoder projection
onto the target logit difference), while dashboard explanations describe their *input* behavior
(the contexts they fire on) — the two can decouple sharply. A recurring special case is the
**suppressor-motif exemplar**: a feature that *fires on concept contexts yet projects against the
concept token* (redundancy suppression under next-token training) — causally ideal for sign-aware
steering, semantically confusing on its dashboard. This step demonstrates those mechanics
directly, and in doing so demos the framework's **graph hydration** capability
(`analysis_backend.hydrate_graph_from_batch`) for detailed post-hoc analysis of a persisted
attribution result:

1. hydrate the step-2 attribution graph; highlight the prompt's concept-token positions;
2. compute each analyzed feature's **input profile** (activation mass at concept positions) and
   **output profile** (signed decoder projection onto the unit target-token unembed difference)
   via the shared `feature_io_profiles` helper;
3. render the decoupling table (signature column flags `decoupled` and `suppressor-motif` rows);
4. project decoder vectors to 2D (UMAP, PCA fallback — tooling shared with the latent-dynamics
   notebooks) with hover details per analyzed feature. Axis tick numbers are intentionally hidden:
   UMAP coordinates are non-metric (arbitrary rotation/scale; only local neighborhood structure is
   meaningful).

Expect roughly 1 of the 5 attribution picks to be input-aligned — a fruit-context feature that is
also a suppressor-motif exemplar (historically L25/16131, with all-`Fruit` negative logits) — and
the rest to be decoupled output machinery.


In [13]:
# @title 5: Decoupling analysis via hydrated graph + UMAP { display-mode: "form" }
import numpy as np

from interpretune.analysis.backends import require_analysis_backend
from it_examples.utils.example_helpers import concept_token_positions, feature_io_profiles
from it_examples.utils.nb_ui_utils import (
    display_concept_positions,
    display_feature_decoupling_table,
    plot_decoder_projection_map,
)

analysis_backend = require_analysis_backend(module)
graph = analysis_backend.hydrate_graph_from_batch(pipeline_results)

# concept-token positions derived from the demo's own concept groups + probe/target tokens
concept_words = {w.lower() for w in (*fruits, *colors, *CONCEPT_TARGET_TOKENS, "orange")}
prompt_token_ids = [int(t) for t in graph.input_tokens]
concept_positions = concept_token_positions(tokenizer, prompt_token_ids, sorted(concept_words))
display_concept_positions(tokenizer, prompt_token_ids, concept_positions)

embed_weight = analysis_backend.get_embedding_weight(module).detach().float().cpu()
target_diff = embed_weight[target_a_id] - embed_weight[target_b_id]
target_diff = target_diff / target_diff.norm()
target_label = f"{CONCEPT_TARGET_TOKENS[0]}-{CONCEPT_TARGET_TOKENS[1]}"

analyzed_pairs = list(dict.fromkeys((int(f[0]), int(f[-1])) for f in steered_features))
all_explanations = dict(feature_explanations)

transcoder_set = getattr(module.replacement_model.transcoders, "_module", module.replacement_model.transcoders)
# J-space overlay (settled surface, #426): folded per-feature signatures from the public
# lens artifacts, so the map colors by J-space disposition. Best-effort: without a resolved
# lens the profiles carry no signatures and the map falls back to output projection.
try:
    from interpretune.analysis.optools import resolve_jlens_layer, resolve_unembed_and_norm_scale

    _overlay_info = resolve_unembed_and_norm_scale(module)
    _, _, _overlay_artifact = resolve_jlens_layer(
        module, {}, {"jlens_model_id": MODEL_NAME, "jlens_layer": JLENS_LAYER}
    )
    _overlay_concept_ids = [int(target_a_id), int(target_b_id)]
    _map_color = "jlens_concept_mass"
except Exception as _overlay_err:
    print(f"[overlay] J-space signatures unavailable: {_overlay_err}")
    _overlay_artifact, _overlay_info, _overlay_concept_ids = None, None, []
    _map_color = "output_projection"
profiles = feature_io_profiles(
    graph,
    analyzed_pairs,
    target_diff,
    transcoder_set,
    concept_positions,
    jlens_artifact=_overlay_artifact,
    unembed_info=_overlay_info,
    tokenizer=tokenizer,
    concept_token_ids=_overlay_concept_ids,
)
display_feature_decoupling_table(profiles, all_explanations, target_label=target_label)

# decoder vectors for the interactive projection map (analyzed + random active-feature background)
rng = np.random.default_rng(17)
active_rows = graph.active_features.cpu()
background_pool = sorted({(int(r[0]), int(r[2])) for r in active_rows} - set(analyzed_pairs))
background_idx = rng.choice(len(background_pool), size=min(300, len(background_pool)), replace=False)
background_pairs = [background_pool[i] for i in background_idx]


def _decoder_rows(pairs):
    return torch.stack(
        [transcoder_set._get_decoder_vectors(lyr, torch.tensor([ft]))[0].detach().float().cpu() for lyr, ft in pairs]
    )


plot_decoder_projection_map(
    profiles,
    _decoder_rows(analyzed_pairs),
    _decoder_rows(background_pairs),
    feature_explanations=all_explanations,
    target_label=target_label,
    title="Steering-feature decoder map",
    color_by=_map_color,
)

Feature,Input concept share,Act mass,Output proj (Fruit-Color),J-lens readout (concept mass),Signature,Explanation
L25/16131,0.481,346.72,-0.2961,(—),suppressor-motif,"the comparison of two varieties of fruit, relating to size, taste, color, and ge"
L24/3865,0.097,191.47,+0.0264,"enfans, avoient, démocr (0.00)",,"dollar signs and other currency symbols, potentially alongside numbers or relate"
L25/13210,0.000,33.75,+0.0227,(—),,grammatical structures and parts of speech like noun phrases and verb phrases
L24/5999,0.347,923.50,+0.0121,"Separator, Interference, Measuring (0.00)",,"language related to institutions, negative situations, the internet, and program"
L24/13277,0.186,1297.00,+0.0061,"Shakspeare, haban, Cæsar (0.00)",,words related to questions and requests


~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## Summary

- Feature-mediated and direct-hook steering paths on one proven example, one backend per run —
  both driven by the same embed-basis concept direction (`concept_basis="store"` builds
  directions from latent-example rows instead and is not used here).
- Semantic grounding via public neuronpedia.org feature dashboards and explanations.
- Input/output decoupling mechanics demonstrated via graph hydration + decoder-space UMAP,
  doubling as a demo of persisted-graph post-hoc analysis.
- Local Neuronpedia dashboards, locally generated explanations, and user-curated feature steering:
  [`ct_concept_steering_demo_local_np.ipynb`](ct_concept_steering_demo_local_np.ipynb).
- Deeper coverage of the underlying op pipeline (per-op invocation, native + hub composition):
  see `ct_analysis_backend_demo.ipynb`. Intervention modes and
  capabilities: `docs/interpretune_intervention_apis.md`.
